# 18 · A computer-use agent that sees and clicks (MicroVM)

A vision loop: **screenshot → model picks an action → mouse/keyboard → repeat**.
The model drives a real virtual desktop inside the VM.

**Key concepts**
- **`sbx.computer_use`** — grouped desktop control: `start()` brings up a virtual
  display (Xvfb); `screenshot.take()` returns PNG bytes; `mouse.move/click`,
  `keyboard.type/press/hotkey` drive input via `xdotool`.
- **Vision tool loop** — feed the screenshot to Bedrock Converse as an image; the
  model returns the next UI action, which we execute, then screenshot again.
- Needs the **`microvm`** backend with a **desktop stack** in the image
  (`Xvfb`, `xdotool`, `scrot`, plus any app) — see notebook 21 to build one.

**Prerequisites**
- AWS creds + region; a **desktop-enabled** image ARN in `RAITBOX_MICROVM_IMAGE`.
  The AWS-managed base is aarch64: build one (notebook 21) with
  `xorg-x11-server-Xvfb` + `xterm` + `ImageMagick` (base repos) and `xdotool`
  from EPEL 9 aarch64. Screenshots use ImageMagick `import` (scrot isn't
  packaged for aarch64). The notebook preflights and skips if tools are absent.
- Bedrock model access for `RAITBOX_BEDROCK_MODEL` (a vision-capable model).

In [ ]:
%pip install -q raitbox

### Configuration

In [ ]:
import base64
import json
import os

import boto3

from raitbox import Sandbox

IMAGE_ARN = os.environ["RAITBOX_MICROVM_IMAGE"]
REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")
MODEL_ID = os.environ.get("RAITBOX_BEDROCK_MODEL", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
bedrock = boto3.client("bedrock-runtime", region_name=REGION)

## 1 · Bring up the desktop and open an app
We start the virtual display and launch a simple GUI target (a terminal emulator)
so there is something on screen to interact with.

In [ ]:
sbx = Sandbox.create(backend="microvm", template=IMAGE_ARN)
print("display:", sbx.computer_use.start())

# Preflight: the desktop tools (xdotool/scrot) must be in the image. They are not
# in the AWS-managed AL2023 base repos, so a purpose-built image is required
# (see notebook 21). We detect them and skip gracefully if absent.
have = sbx.commands.run(
    "command -v xdotool >/dev/null && "
    "{ command -v scrot >/dev/null || command -v import >/dev/null; } "
    "&& echo yes || echo no"
).stdout.strip()
DESKTOP_READY = have == "yes"
print("desktop tools present:", DESKTOP_READY)

if DESKTOP_READY:
    sbx.commands.run("DISPLAY=:99 xterm -geometry 100x30+10+10 & sleep 2", background=True)
    print("screen:", sbx.computer_use.display.get_info())
else:
    print("Skipping the live loop: this image lacks xdotool + a capture tool (scrot/ImageMagick). Build a desktop "
          "image (notebook 21) whose base provides them to run computer use for real.")

## 2 · The action loop

In [ ]:
TOOLS = {"tools": [{"toolSpec": {
    "name": "computer",
    "description": "Control the desktop.",
    "inputSchema": {"json": {"type": "object", "properties": {
        "action": {"type": "string", "enum": ["move", "click", "type", "key", "done"]},
        "x": {"type": "integer"}, "y": {"type": "integer"},
        "text": {"type": "string"}, "key": {"type": "string"},
    }, "required": ["action"]}},
}}]}


def do(action):
    a = action["action"]
    if a == "move":
        sbx.computer_use.mouse.move(action["x"], action["y"])
    elif a == "click":
        sbx.computer_use.mouse.click(action.get("x"), action.get("y"))
    elif a == "type":
        sbx.computer_use.keyboard.type(action["text"])
    elif a == "key":
        sbx.computer_use.keyboard.press(action["key"])
    return a


def screenshot_block():
    png = sbx.computer_use.screenshot.take()
    return {"image": {"format": "png", "source": {"bytes": png}}}

In [ ]:
if DESKTOP_READY:
    TASK = ("Click into the terminal window, type: echo hello-from-raitbox  and press "
            "Return. Call computer with action=done when finished.")
    messages = [{"role": "user", "content": [{"text": TASK}, screenshot_block()]}]
    for step in range(6):
        resp = bedrock.converse(
            modelId=MODEL_ID, messages=messages, toolConfig=TOOLS,
            inferenceConfig={"maxTokens": 1024, "temperature": 0},
        )
        out = resp["output"]["message"]
        messages.append(out)
        tool_uses = [b["toolUse"] for b in out["content"] if "toolUse" in b]
        if not tool_uses:
            break
        results = []
        stop = False
        for tu in tool_uses:
            act = tu["input"]
            name = do(act)
            print("step", step, "->", json.dumps(act))
            stop = stop or name == "done"
            results.append({"toolResult": {"toolUseId": tu["toolUseId"],
                            "content": [{"text": f"did {name}"}, screenshot_block()]}})
        messages.append({"role": "user", "content": results})
        if stop:
            break
else:
    print("(loop skipped — no desktop tools)")

## 3 · Verify what landed on screen

In [ ]:
if DESKTOP_READY:
    final = sbx.computer_use.screenshot.take()
    with open("computer_use_final.png", "wb") as f:
        f.write(final)
    print("saved screenshot:", len(final), "bytes -> computer_use_final.png")
    sbx.computer_use.stop()

sbx.kill()
print("done")

## Recap
- `computer_use` turned screenshots + a tool schema into a see-think-act loop.
- The same primitives (`mouse`, `keyboard`, `screenshot`, `display`) cover
  browser automation, form filling, and GUI testing.

**Related:** notebook 21 (build a desktop image), `docs/CONCEPTS.md` (Computer use).